# Survival: the Mayo model for primary biliary cirrhosis

Primary biliary cirrhosis is a slow autoimmune disease of the liver, and in the 1980s a liver transplant was its only cure. When should a patient be put forward for one? Dickson, Grambsch, Fleming, Fisher and Langworthy (1989) answered with a Cox model built from the 312 patients of two Mayo Clinic trials: five measurements that need only a blood sample and an examination, no liver biopsy, combined into a risk score R that gives each patient's chance of surviving 1 to 7 years. They tested it on 106 patients who had not been used to build it, refitted it on all 418, and printed the tables a doctor needs to use it. The values of their Tables 2 and 4, most of Table 3, and their two worked patients are printed beside Julia's.

The data is `pbc`, from PollisDatasets: the 418 Mayo patients as Fleming and Harrington (1991) published them. The Cox model is fitted twice: with Survival.jl, which handles patients who die on the same day by Efron's method, and with a short hand-written fit that handles them by Breslow's, as the paper's software did.

In [ ]:
Pollis.packages("Survival", "ForwardDiff", "StatsBase", "Distributions", "PollisDatasets", "Plots")

## Set-up

The follow-up `days` and the event `dead` of each patient, the trial patients `trial`, who build the model, and the others, `other`; `riskfactors`, the five measurements of the model; and `replicate`, which prints each value in full, rounded to the published precision, beside the published value, and how many decimals they share.

In [ ]:
using Survival, ForwardDiff, StatsBase, Distributions, PollisDatasets, Plots, Printf, Statistics, LinearAlgebra

"""
	replicate(title, rows)

Print each Julia value in full, rounded to the published precision and beside the published value, with the number of decimals they share.
Each row is `(label, value, published)`, with `published` written as in the source: its decimals give the published precision.
"""
function replicate(title, rows)
	@printf("\n%s\n%-26s%16s%14s%14s\n", title, "", "Julia", "rounded", "published")
	for (label, value, published) in rows
		p = parse(Float64, replace(published, "," => ""))
		d = occursin('.', published) ? length(split(published, '.')[end]) : 0  # decimals as published
		k = findlast(j -> round(value, digits=j) == round(p, digits=j), 0:d)  # position in 0:d
		@printf("%-26s%16.8f%14s%14s  %s\n", label, value, Printf.format(Printf.Format("%.$(d)f"), value), published, k === nothing ? "DIFFERENT" : "same to $(k - 1) decimals")
	end
end

any(row -> row.id == "pbc", datasets()) || error("This workflow needs the pbc dataset: update PollisDatasets")
pbc = dataset("pbc");  # one row per patient
days = Float64.(pbc.time);  # from entry to death, transplantation or the end of follow-up (July 1986)
dead = pbc.status .== 2;  # death from any cause; transplanted patients are censored at transplantation (p. 2)
trial = 1:312;  # the two D-penicillamine trials: they build the model
other = 313:418;  # eligible for the trials but declined: they test it
months = 365.25 / 12;
names5 = ["log bilirubin", "log albumin", "age", "log prothrombin time", "edema"];  # the order of Table 2

"""
	riskfactors(rows; protime=pbc.protime)

The five measurements of the Mayo model for the patients in `rows`, one column each, in the order of Table 2.
"""
riskfactors(rows; protime=pbc.protime) = Float64.(hcat(log.(pbc.bili[rows]), log.(pbc.albumin[rows]), pbc.age[rows], log.(protime[rows]), pbc.edema[rows]))

X = riskfactors(trial);

## 1. The patients

The trial patients entered from January 1974 to May 1984 and were followed until July 1986: 125 died, 19 had a transplant, and the rest were alive or lost to follow-up (Dickson et al. 1989, p. 2). Both trials found D-penicillamine no better than placebo, so the treatment is ignored and the trials are pooled as the natural history of the disease. The declined patients are similar at entry (Table 3, p. 5); two of them lack a prothrombin time (Section 5). Bilirubin is reported as a geometric mean, since the model uses its logarithm.

In [ ]:
outcome(rows, k) = sum(pbc.status[rows] .== k)  # k: 0 censored, 1 transplant, 2 dead
medianmonths(rows, k) = median(days[rows][pbc.status[rows] .== k]) / months
replicate("Follow-up to July 1986 (Dickson et al. 1989, p. 2)", [
	("trial: deaths", outcome(trial, 2), "125"), ("  median months", medianmonths(trial, 2), "39"),
	("trial: transplants", outcome(trial, 1), "19"), ("  median months", medianmonths(trial, 1), "47"),
	("declined: deaths", outcome(other, 2), "36"), ("  median months", medianmonths(other, 2), "26"),
	("declined: transplants", outcome(other, 1), "6"), ("  median months", medianmonths(other, 1), "50"),
	("declined: alive", outcome(other, 0), "64"), ("  median months", medianmonths(other, 0), "54")])

geomean(x) = exp(mean(log.(x)))
stage(rows, s) = 100 * mean(skipmissing(pbc.stage[rows]) .== s)  # % of the patients with a biopsy
for (label, rows, published) in [("trial", trial, ["50.0", "1.78", "3.52", "10.73", "5", "22", "38", "35"]),
		("declined", other, ["52.9", "1.75", "3.43", "10.75", "5", "25", "35", "35"])]
	replicate("$label patients at entry (Table 3, p. 5)", collect(zip(
		["age", "bilirubin (geometric)", "albumin", "prothrombin time", "stage 1, %", "stage 2, %", "stage 3, %", "stage 4, %"],
		[mean(pbc.age[rows]), geomean(pbc.bili[rows]), mean(pbc.albumin[rows]), mean(skipmissing(pbc.protime[rows])), stage.(Ref(rows), 1:4)...],
		published)))
end
replicate("Trial patients at entry (p. 3)", [("cirrhosis (stage 4), %", stage(trial, 4), "35"),
	("albumin below 3.0, %", 100 * mean(pbc.albumin[trial] .< 3), "9"), ("bilirubin above 5, %", 100 * mean(pbc.bili[trial] .> 5), "18")])

km = fit(KaplanMeier, days[trial], dead[trial]);
plot(km.events.time ./ 365.25, km.survival, linetype=:steppost, label="Kaplan-Meier, 312 trial patients",
	xlabel="years since entry", ylabel="survival", ylims=(0, 1))

## 2. The Cox model with Survival.jl

The model gives each patient a risk score R = β₁ X₁ + … + βₖ Xₖ: a patient whose R is larger by d has exp(d) times the risk of dying, at every moment (p. 2). Of 45 candidate variables, the authors kept the five that need no biopsy and were significant at 1% in both forward and backward stepwise selection: log bilirubin, log albumin, age, log prothrombin time and edema, coded 0 (none), 0.5 (untreated, or resolved with diuretics) or 1 (despite diuretics) (pp. 3-4).

Survival.jl fits the model on the 312 trial patients. Several patients died on the same day, and Survival.jl handles such ties by Efron's (1977) method: the coefficients agree with Table 2 to two or three digits, not to all four.

In [ ]:
efron = fit(CoxModel, X, EventTime.(days[trial], dead[trial]))
@printf("deaths on a day when another trial patient also died: %d of %d\n", sum(dead[trial]) - length(unique(days[trial][dead[trial]])), sum(dead[trial]))
replicate("Survival.jl, Efron's ties (Table 2, initial Mayo model, p. 4)", collect(zip(names5, coef(efron), ["0.8792", "-3.053", "0.0333", "3.016", "0.7847"])))

## 3. Breslow's ties: the published model

The paper's fits were made with the SAS procedure PHGLM (p. 3), which handles ties by Breslow's (1974) method: all patients who die on the same day share one risk set. Survival.jl has no option for it, so `cox` writes Breslow's partial likelihood out and maximises it by Newton's method, with exact derivatives from ForwardDiff. The coefficients, standard errors, chi-squares (the squared z statistics) and p values then agree with the first column of Table 2; the p values printed as < 0.0001 are below that bound too.

In [ ]:
"""
	partial(β, X, t, e)

Breslow's (1974) partial log likelihood of the Cox model: the patients who die on the same day share one risk set.
"""
function partial(β, X, t, e)
	η = X * β
	sum(η[i] - log(sum(exp.(η[t .>= t[i]]))) for i in findall(e))
end

"""
	cox(X, t, e)

Fit the Cox model by Newton's method on Breslow's partial likelihood: coefficients, standard errors, Wald chi-squares and their p values,
and the likelihood ratio chi-square of the whole model.
"""
function cox(X, t, e)
	f(β) = partial(β, X, t, e)
	β = zeros(size(X, 2))
	for _ in 1:25
		β -= ForwardDiff.hessian(f, β) \ ForwardDiff.gradient(f, β)
	end
	se = sqrt.(diag(inv(-ForwardDiff.hessian(f, β))))
	chi2 = (β ./ se) .^ 2
	(β = β, se = se, chi2 = chi2, p = ccdf.(Chisq(1), chi2), LR = 2 * (f(β) - f(zero(β))), X = X, t = t, e = e)
end

"""
	table2(title, m, published)

Replicate a column of Table 2: `published` holds the coefficient, standard error, chi-square and p value of each variable, as printed,
with `nothing` for a p value printed as < 0.0001.
"""
function table2(title, m, published)
	rows = []
	for (i, (b, se, chi2, p)) in enumerate(published)
		push!(rows, (names5[i], m.β[i], b), ("  se", m.se[i], se), ("  chi-square", m.chi2[i], chi2))
		p === nothing ? @assert(m.p[i] < 0.0001) : push!(rows, ("  p", m.p[i], p))
	end
	replicate(title, rows)
end

initial = cox(X, days[trial], dead[trial]);
table2("Initial Mayo model, 312 patients (Table 2, p. 4)", initial, [("0.8792", "0.0987", "79.30", nothing), ("-3.053", "0.724", "17.78", nothing),
	("0.0333", "0.0087", "14.76", "0.0001"), ("3.016", "1.024", "8.68", "0.0032"), ("0.7847", "0.2991", "6.88", "0.0087")])
replicate("The whole model (p. 4)", [("likelihood ratio chi-square", initial.LR, "199.2")])

## 4. Does the model fit?

The authors ranked the trial patients by R and cut them into five groups of 25 deaths each, then compared each group's Kaplan-Meier curve with its predicted curve, the average of its patients' predicted survival (Fig. 1, p. 4). The lowest-risk fifth had a better than 96% chance of surviving 5 years, the highest-risk fifth a worse than 52% chance of surviving 1 year. A patient's predicted survival is S(t) = S₀(t)^exp(R − c), where S₀ is the survival of a patient with the mean risk score c, estimated, as in the paper, by Kalbfleisch and Prentice's (1980) method (p. 3).

In [ ]:
"""
	survival(m)

The predicted survival S(t, R) = S0(t)^exp(R - c) of a fitted model, at t days for a patient with risk score R, where c is the mean
risk score and S0 is Kalbfleisch and Prentice's (1980) estimate: a product over the days with deaths of factors α, each the root of
Σ_deaths w / (1 - α^w) = Σ_at risk w, with w = exp(R - c).
"""
function survival(m)
	R = m.X * m.β
	c = mean(R)
	w = exp.(R .- c)
	times = sort(unique(m.t[m.e]))
	α = map(times) do s
		deaths = findall(m.e .& (m.t .== s))
		g(a) = sum(w[i] / (1 - a^w[i]) for i in deaths) - sum(w[m.t .>= s])  # increasing in a
		lo, hi = 0.0, 1.0
		for _ in 1:100
			mid = (lo + hi) / 2
			g(mid) > 0 ? (hi = mid) : (lo = mid)
		end
		(lo + hi) / 2
	end
	S0 = cumprod(α)
	(t, r) -> (j = searchsortedlast(times, t); j == 0 ? 1.0 : S0[j]^exp(r - c))
end
kmat(km, t) = (j = searchsortedlast(km.events.time, t); j == 0 ? 1.0 : km.survival[j])  # a Kaplan-Meier curve at t days

S = survival(initial);
R = X * initial.β;
order = sortperm(R);
deaths = cumsum(dead[trial][order]);
quintiles = [order[(k == 1 ? 1 : findfirst(==(25(k - 1)), deaths) + 1):(k == 5 ? 312 : findfirst(==(25k), deaths))] for k in 1:5];

@printf("%-10s%6s%8s%14s%12s%14s%12s\n", "quintile", "n", "deaths", "1 year: K-M", "predicted", "5 years: K-M", "predicted")
plt = plot(xlabel="years since entry", ylabel="survival", ylims=(0, 1), legend=:bottomleft)
for (k, g) in enumerate(quintiles)
	kmg = fit(KaplanMeier, days[trial][g], dead[trial][g])
	predicted(t) = mean(S(t, R[i]) for i in g)
	@printf("%-10d%6d%8d%14.3f%12.3f%14.3f%12.3f\n", k, length(g), sum(dead[trial][g]), kmat(kmg, 365.25), predicted(365.25), kmat(kmg, 5 * 365.25), predicted(5 * 365.25))
	grid = 0:15:8 * 365.25
	plot!(plt, grid ./ 365.25, kmat.(Ref(kmg), grid), linetype=:steppost, linestyle=:dash, color=k, label="")
	plot!(plt, grid ./ 365.25, predicted.(grid), color=k, label="quintile $k")
end
@printf("\nquintile 1, Kaplan-Meier 5-year survival: %.3f (published: > .96)\n", kmat(fit(KaplanMeier, days[trial][quintiles[1]], dead[trial][quintiles[1]]), 5 * 365.25))
@printf("quintile 5, Kaplan-Meier 1-year survival: %.3f (published: < .52)\n", kmat(fit(KaplanMeier, days[trial][quintiles[5]], dead[trial][quintiles[5]]), 365.25))
plt

## 5. Testing it on patients it has not seen

The 106 declined patients were not used to fit the model. Their missing prothrombin times were estimated by regression, as the paper did (p. 3); here log prothrombin time is regressed on the other four measurements of the model. The spread of their risk scores is close to that of the trial patients (Table 3).

In [ ]:
missingpt = findall(ismissing, pbc.protime)  # two declined patients
known = setdiff(1:418, missingpt);
Z(rows) = Float64.(hcat(ones(length(rows)), log.(pbc.bili[rows]), log.(pbc.albumin[rows]), pbc.age[rows], pbc.edema[rows]))
b = Z(known) \ log.(Float64.(pbc.protime[known]));  # least squares
protime = Float64.(coalesce.(pbc.protime, NaN));
protime[missingpt] = exp.(Z(missingpt) * b);
@printf("estimated prothrombin times: %s seconds\n", join(round.(protime[missingpt], digits=2), ", "))

Xall = riskfactors(1:418; protime=protime);
V = Xall[other, :];
Rv = V * initial.β;
replicate("Risk score R, initial Mayo model (Table 3, p. 5)", [
	("trial: 10th percentile", quantile(R, 0.1), "4.03"), ("  50th percentile", quantile(R, 0.5), "5.24"), ("  90th percentile", quantile(R, 0.9), "7.67"),
	("declined: 10th percentile", quantile(Rv, 0.1), "4.24"), ("  50th percentile", quantile(Rv, 0.5), "5.41"), ("  90th percentile", quantile(Rv, 0.9), "7.45")])

### Does the model predict deaths it was not fitted to?

A model always fits the patients it was estimated on better than it will fit new ones: the coefficients have adapted to that sample's chance features. The honest test is on patients the model has never seen, and the 106 declined patients are such a sample. The authors ranked them by risk, cut them into three groups with roughly equal numbers of deaths, and compared the deaths observed in each group with the deaths the model predicts, by one-sample log-rank tests (pp. 3 and 5):

| Risk group | p |
|---|---|
| low | 0.53 |
| medium | 0.69 |
| high | 0.57 |

None of the three tests rejects the model. The authors add two cautions: the test treats the model's predicted survival as fixed when it is itself estimated, so it rejects more often than its nominal level (p. 3); and a model may be expected to do well on patients from its own clinic (p. 5).

**These p-values cannot be replicated from the public data.** The groups were formed from the mean of two ranks per patient, one by this model's risk score and one by the score of another published model, which needs central cholestasis, read from the liver biopsy and not in the data (p. 3); nor does the paper give the size of each group. So the groups cannot be rebuilt.

What can be shown is the test itself. The cell below cuts the declined patients into three groups with equal numbers of deaths by the Mayo score alone, so these are not the paper's groups, and its p-values are not to be compared with the table. A patient's expected deaths are the cumulative hazard the model predicts up to the end of his or her follow-up, −log S(t, R).

In [ ]:
tv, ev = days[other], dead[other];
order = sortperm(Rv);
deaths = cumsum(ev[order]);
cuts = [findfirst(>=(round(sum(ev) * k / 3)), deaths) for k in 1:2];
thirds = [order[1:cuts[1]], order[cuts[1]+1:cuts[2]], order[cuts[2]+1:end]];
for (label, g) in zip(["low risk", "medium risk", "high risk"], thirds)
	observed = sum(ev[g])
	expected = sum(-log(S(tv[i], Rv[i])) for i in g)
	p = ccdf(Chisq(1), (observed - expected)^2 / expected)
	@printf("%-12s %3d patients, deaths observed %2d, expected %5.1f, p = %.2f\n", label, length(g), observed, expected, p)
end

## 6. The final Mayo model, on all 418 patients

Since the model predicted the declined patients well, the authors pooled all 418 patients and refitted it (Table 2, last column, p. 4): the coefficients change little and the standard errors shrink.

In [ ]:
final = cox(Xall, days, dead);
table2("Final Mayo model, 418 patients (Table 2, p. 4)", final, [("0.8707", "0.0826", "111.03", nothing), ("-2.533", "0.648", "15.27", "0.0001"),
	("0.0394", "0.0077", "26.53", nothing), ("2.380", "0.767", "9.64", "0.0019"), ("0.8592", "0.2711", "10.04", "0.0015")])

## 7. Using the model

The paper gives a doctor the model in two pieces (p. 6). First, R from rounded coefficients; then S(t) = S₀(t)^exp(R − 5.07), where S₀(t) is the survival of a patient with R = 5.07, the mean over the 418 patients (Table 4, p. 6). Its two worked patients: a 52-year-old with bilirubin 0.5, albumin 4.5, prothrombin time 10.1 and no edema has R = 3.12 and a 96% chance of surviving 5 years; one with bilirubin 13.9, albumin 2.8, prothrombin time 13.8 and edema treated with diuretics has R = 8.39 and a 43% chance of surviving 1 year, and would be considered for a transplant.

Here S₀ comes from the Julia fit, at the mean of its own risk scores, which is what 5.07 is for the published coefficients.

In [ ]:
mayo(bilirubin, albumin, age, protime, edema) = 0.871log(bilirubin) - 2.53log(albumin) + 0.039age + 2.38log(protime) + 0.859edema  # p. 6
Sf = survival(final);
Rmean = mean(Xall * final.β);
replicate("Using the model (p. 6)", [
	("mean R, 418 patients", mean(mayo.(pbc.bili, pbc.albumin, pbc.age, protime, pbc.edema)), "5.07"),
	("patient 1: R", mayo(0.5, 4.5, 52, 10.1, 0), "3.12"), ("  5-year survival", 0.774^exp(mayo(0.5, 4.5, 52, 10.1, 0) - 5.07), ".96"),
	("patient 2: R", mayo(13.9, 2.8, 52, 13.8, 0.5), "8.39"), ("  1-year survival", 0.970^exp(mayo(13.9, 2.8, 52, 13.8, 0.5) - 5.07), ".43")])
replicate("Baseline survival S0(t) (Table 4, p. 6)", [("year $y", Sf(y * 365.25, Rmean), p) for (y, p) in
	zip(1:7, ["0.970", "0.941", "0.883", "0.833", "0.774", "0.721", "0.651"])])

patients = [("patient 1", [log(0.5), log(4.5), 52, log(10.1), 0]), ("patient 2", [log(13.9), log(2.8), 52, log(13.8), 0.5])];
grid = 0:15:7 * 365.25
plot(xlabel="years", ylabel="predicted survival", ylims=(0, 1))
for (label, x) in patients
	plot!(grid ./ 365.25, Sf.(grid, dot(x, final.β)), linetype=:steppost, label=label)
end
plot!()

## 8. The branch: does a biopsy add anything?

The Mayo model was built to avoid a liver biopsy, and the European and Yale models it competed with both needed one (Table 5, p. 6). Is histologic stage, which only a biopsy gives, worth adding? It is strongly associated with R (Kendall's τ of .38, p. 6), so most of what it says is already in R. Adding it to the initial model, the paper's own rule for keeping a variable, significance at 1% (p. 3), decides: if `biopsy` is false, the five-variable model stands.

In [ ]:
withstage = cox(hcat(X, Float64.(pbc.stage[trial])), days[trial], dead[trial]);
lr = withstage.LR - initial.LR;
replicate("Histologic stage and the Mayo risk score (p. 6)", [("Kendall's tau", corkendall(Float64.(pbc.stage[trial]), mayo.(pbc.bili[trial], pbc.albumin[trial], pbc.age[trial], pbc.protime[trial], pbc.edema[trial])), ".38")])
@printf("\nadding stage to the initial model: likelihood ratio chi-square %.2f, p = %.4f\n", lr, ccdf(Chisq(1), lr))
biopsy = ccdf(Chisq(1), lr) <= 0.01  # the rule: significant at 1%, as for the five variables

## 9. What differs, and why

Most values agree to the published precision: the whole first column of Table 2, all seven years of Table 4, and both worked patients. The rest are listed here, explained where the cause was found.

- **Survival.jl's coefficients (Section 2).** Efron's ties instead of Breslow's: with Breslow's, every value of the column agrees (Section 3).
- **The likelihood ratio chi-square (199.1 against 199.2).** Breslow's ties give 199.13, Efron's 199.3. Not explained.
- **The final model, in the third decimal (e.g. log prothrombin time 2.379 against 2.380).** The paper estimated missing values for the 16 declined patients who lacked prothrombin time, cirrhosis or cholestasis, with regressions fitted to the 90 complete patients after backward elimination (p. 3). The public file lacks only two prothrombin times and has no cholestasis variable, so the regression here is simpler. Setting the two missing times to anything from 9 to 13 seconds moves the coefficients in the third decimal: the gaps are of that size.
- **The mean risk score (5.05 against 5.07).** It also depends on those estimated values. Not proven.
- **Stage 2 of the trial patients (21.5% against 22%).** 67 of the 312 have stage 2; 22% needs 68. Not explained: the other stages agree.
- **The 10th and 90th percentiles of R (Table 3).** In the tails, neighbouring patients' scores are far apart: the trial's 90th percentile lies between 7.621 and 7.692, the declined patients' 10th and 90th between 4.157 and 4.273, and between 7.369 and 7.628. Every published value is inside its interval, so a different percentile definition gives it.
- **Kendall's tau (.37 against .38).** This is tau-b on the trial patients; the paper says neither which tau nor which patients (all patients with a biopsy give .34). Not explained.

**Now in Compose › New Methods.** Survival.jl fits the Cox model with Efron's ties only, and has no predicted survival curves for it. The New Methods notebook **The Cox model with Breslow's ties, and its predicted survival** writes both out (`cox` and `survival`), tested against Tables 2 and 4 of this paper. Open it from **Compose › New Methods**. The one-sample log-rank test of Section 5 is not among them: its groups use a biopsy variable missing from the public data, so there are no published values to test it against.

## References

- Breslow, N. (1974). Covariance analysis of censored survival data. *Biometrics* 30(1), 89-99.
- Dickson, E. R., Grambsch, P. M., Fleming, T. R., Fisher, L. D. and Langworthy, A. (1989). Prognosis in primary biliary cirrhosis: model for decision making. *Hepatology* 10(1), 1-7. doi:10.1002/hep.1840100102
- Efron, B. (1977). The efficiency of Cox's likelihood function for censored data. *Journal of the American Statistical Association* 72(359), 557-565.
- Fleming, T. R. and Harrington, D. P. (1991). *Counting Processes and Survival Analysis*. Wiley. Appendix D: the data.
- Kalbfleisch, J. D. and Prentice, R. L. (1980). *The Statistical Analysis of Failure Time Data*. Wiley.
- Kaplan, E. L. and Meier, P. (1958). Nonparametric estimation from incomplete observations. *Journal of the American Statistical Association* 53(282), 457-481.
- Data: the 418 patients as distributed with the R package survival (`pbc`).